# dataloader-pin-memory-workers — ex2: DataLoader with shuffle=True + seeded worker_init_fn — verify batch shapes and full-epoch coverage

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataloader-pin-memory-workers`. Running the final beacon cell reports progress against the `PyTorch: DataLoader pin_memory + workers` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader pin_memory + workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-pin-memory-workers`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-pin-memory-workers"
DD_SUBTOPIC = "PyTorch: DataLoader pin_memory + workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DataLoader + `worker_init_fn` + shuffle — batch shape stays correct

Ex1 built a DataLoader with `num_workers` and `pin_memory` configured. The deepening move shows that under `shuffle=True` with multiple workers, batch shapes (and batch counts) remain correct as long as the main-process seed is set AND each worker gets a deterministic seed via `worker_init_fn`.

```python
def worker_init_fn(worker_id):
    base = torch.initial_seed() % (2**32)
    np.random.seed(base + worker_id)
    random.seed(base + worker_id)
```

**Why this matters.** `torch.initial_seed()` inside a worker returns the per-worker base seed PyTorch assigns. Without re-seeding numpy and Python `random`, every worker draws the same numpy/random stream — a silent source of duplicate augmentations in data-augmentation pipelines.

**Shape invariant.** Regardless of shuffle / workers / pin_memory, the DataLoader still yields `(batch_size, *item_shape)` tensors and the total number of items emitted across one epoch equals `len(dataset)`. `pin_memory=False` on CPU; the shape contract is unchanged.

**CPU-only is fine.** `pin_memory=True` requires CUDA at runtime. In a CPU test we pass `pin_memory=False` and still exercise the rest of the config (workers, shuffle, seeded init).

### Exercise 2 — DataLoader with shuffle=True + seeded worker_init_fn — verify batch shapes and full-epoch coverage

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `worker_init_fn` + `torch.initial_seed()` to seed each worker's numpy/random RNG deterministically, and verify that under `shuffle=True` the resulting DataLoader still yields `(batch_size, *item_shape)` batches and covers every dataset item in one epoch.
> Keywords: dataloader, worker_init_fn, shuffle, batch-shape
> ```

**KCs targeted:** `worker-init-fn-seeded-rng`, `shuffle-preserves-batch-shape`

Implement `ex2_make_seeded_shuffle_dataloader(dataset, batch_size, num_workers)`. Returns a `DataLoader` configured for:

1. `batch_size` items per batch.
2. `num_workers` worker processes (may be 0 for in-main).
3. `shuffle=True`.
4. `pin_memory=False` (CPU-only test environment).
5. `drop_last=False` — keep the last partial batch.
6. `worker_init_fn` that seeds numpy AND Python `random` per worker via `torch.initial_seed()`:
```python
def worker_init_fn(worker_id):
    import numpy as np, random
    base = torch.initial_seed() % (2**32)
    np.random.seed(base + worker_id)
    random.seed(base + worker_id)
```

Return the configured DataLoader.

**The test then verifies:**
- batch shape `(B, *item_shape)`, including last batch when `len(dataset) % batch_size != 0`.
- total items across one epoch == `len(dataset)`.
- two epochs over the same loader yield DIFFERENT orderings (shuffle works) but the same total item count.

**Use `from torch.utils.data import DataLoader, TensorDataset`.**

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def ex2_make_seeded_shuffle_dataloader(dataset, batch_size: int, num_workers: int) -> DataLoader:
    """Build a DataLoader with shuffle + seeded worker_init_fn (CPU-only)."""
    raise NotImplementedError()


def _test_ex2():
    from torch.utils.data import TensorDataset, DataLoader

    # === Construction ===
    t.manual_seed(0)
    data = t.randn(11, 3, 4)   # 11 items, shape (3, 4); 11 is intentionally non-divisible
    labels = t.arange(11)
    ds = TensorDataset(data, labels)
    loader = ex2_make_seeded_shuffle_dataloader(ds, batch_size=4, num_workers=0)
    assert isinstance(loader, DataLoader), f'must return a DataLoader, got {type(loader).__name__}'

    # === Config readback ===
    assert loader.batch_size == 4
    assert loader.num_workers == 0
    assert loader.pin_memory is False, 'pin_memory must be False on CPU'
    assert loader.drop_last is False
    # Sampler must be a RandomSampler (shuffle=True).
    from torch.utils.data import RandomSampler
    assert isinstance(loader.sampler, RandomSampler), f'shuffle=True needs RandomSampler, got {type(loader.sampler).__name__}'
    # worker_init_fn must be set and callable.
    assert callable(loader.worker_init_fn), 'worker_init_fn must be set'

    # === Batch shape invariant under shuffle ===
    all_items = []
    batch_shapes = []
    for x_batch, y_batch in loader:
        batch_shapes.append(tuple(x_batch.shape))
        # Each non-last batch: (4, 3, 4). Last: (3, 3, 4).
        assert x_batch.shape[1:] == (3, 4), f'item dims must be (3, 4), got {x_batch.shape[1:]}'
        assert x_batch.shape[0] in (4, 3), f'batch size must be 4 or 3 (last partial), got {x_batch.shape[0]}'
        all_items.append(y_batch)

    # === Total items covered ===
    flat = t.cat(all_items)
    assert len(flat) == 11, f'one epoch must yield all 11 items, got {len(flat)}'
    assert set(flat.tolist()) == set(range(11)), 'every dataset index must appear exactly once'

    # === Last batch is partial: 11 = 4 + 4 + 3 ===
    assert sorted([s[0] for s in batch_shapes]) == [3, 4, 4], f'expected batches of sizes [3,4,4], got {batch_shapes}'

    # === Two epochs: shuffle changes order, but total count stays 11 ===
    t.manual_seed(42)
    order_a = t.cat([y for _, y in loader]).tolist()
    t.manual_seed(43)
    order_b = t.cat([y for _, y in loader]).tolist()
    assert sorted(order_a) == sorted(order_b) == list(range(11)), 'both epochs cover full dataset'
    assert order_a != order_b, f'two epochs with different seeds should differ; got identical {order_a}'

    # === worker_init_fn actually seeds numpy ===
    # Direct call to verify the function reseeds numpy deterministically.
    wif = loader.worker_init_fn
    # Set torch initial_seed via manual_seed (which sets initial_seed in the current thread).
    t.manual_seed(12345)
    wif(0)
    a0 = np.random.rand()
    t.manual_seed(12345)
    wif(0)
    a1 = np.random.rand()
    assert a0 == a1, f'same torch seed + same worker_id must produce same numpy stream; got {a0} vs {a1}'
    t.manual_seed(12345)
    wif(1)
    a2 = np.random.rand()
    assert a0 != a2, f'different worker_id should produce different numpy stream; got identical {a0}'

    # === A divisible batch_size (8 % 4 == 0) → no partial batch ===
    ds_div = TensorDataset(t.randn(8, 2), t.arange(8))
    loader_div = ex2_make_seeded_shuffle_dataloader(ds_div, batch_size=4, num_workers=0)
    shapes = [b[0].shape[0] for b in loader_div]
    assert shapes == [4, 4], f'divisible: expected [4, 4], got {shapes}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
from torch.utils.data import DataLoader
import torch as _t
import numpy as _np
import random as _random

def _worker_init_fn(worker_id):
    base = _t.initial_seed() % (2 ** 32)
    _np.random.seed(base + worker_id)
    _random.seed(base + worker_id)

def ex2_make_seeded_shuffle_dataloader(dataset, batch_size, num_workers):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        pin_memory=False,
        drop_last=False,
        worker_init_fn=_worker_init_fn,
    )
```

**`torch.initial_seed()` inside a worker is the per-worker base seed PyTorch already assigned.** Calling it from a worker_init_fn lets you derive a numpy/random seed that's distinct per worker AND deterministic given `torch.manual_seed(...)` in the main process.

**`% (2**32)` is required.** numpy and Python's `random` both want a uint32 seed. `torch.initial_seed()` returns a 63-bit int; the modulo collapses it safely.

**`pin_memory=False` on CPU is correct.** `pin_memory=True` would raise at first batch fetch without CUDA. The contract we're exercising is the rest of the DataLoader config — workers, shuffle, seeded init — all of which work fine CPU-only.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()